In [2]:
import requests
import json
import time
import os
import logging

In [16]:
BASE_URL="https://openlibrary.org/search.json"

In [17]:
TITLES = [
    "harry potter",
    "the hobbit",
    "percy jackson",
    "lord of the rings",
    "the hunger games",
    "chronicles of narnia",
]

In [18]:
PAGES_PER_TITLE = 10
OUTPUT_FOLDER = "Search"
DELAY_BETWEEN_REQUESTS = 0.5
LOG_FILE = "fetch_openlibrary_books.log"

In [ ]:
def setup_logging():
    logger = logging.getLogger("openlibrary_fetcher")
    logger.setLevel(logging.INFO) 
    formatter = logging.Formatter(
        fmt="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
 
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)
 
    file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
 
    return logger

In [ ]:
logger = setup_logging()

In [20]:
def fetch_page(title, page):
    params = {
        "title": title,
        "page": page
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        response.raise_for_status()  
        return response.json()
    except requests.exceptions.RequestException as error:
        logger.error(f"Error fetching '{title}' page {page}: {error}")
        return None

In [21]:
def make_title_folder_name(title):
    safe_title = title.lower().replace(" ", "_")
    return f"Title={safe_title}"

In [22]:
def make_page_folder_name(page):
    return f"Page={page:02d}"

In [23]:
def save_page_to_json_file(data, title, page, base_folder):
    title_folder = make_title_folder_name(title)
    page_folder = make_page_folder_name(page)
    folder_path = os.path.join(base_folder, title_folder, page_folder)
    os.makedirs(folder_path, exist_ok=True)
    filepath = os.path.join(folder_path, "data.json")
    with open(filepath, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
 
    logger.info(f"Saved to {filepath}")

In [ ]:
def fetch_and_save_all_pages_for_title(title, num_pages, base_folder):
    for page in range(1, num_pages + 1):
        logger.info(f"Fetching '{title}' - page {page}/{num_pages}...")
        data = fetch_page(title, page)
 
        if data is not None:
            save_page_to_json_file(data, title, page, base_folder)
        else:
            logger.warning(f"Skipped saving '{title}' page {page} (request failed)")
 
        time.sleep(DELAY_BETWEEN_REQUESTS)

In [ ]:
def main():
    logger.info("Starting OpenLibrary data download...")
 
    for title in TITLES:
        logger.info(f"Searching for: '{title}'")
 
        # Fetch every page for this title, saving each one as we go
        fetch_and_save_all_pages_for_title(title, PAGES_PER_TITLE, OUTPUT_FOLDER)
 
    logger.info(f"Check the '{OUTPUT_FOLDER}' folder for your files.")

In [28]:
if __name__ == "__main__":
    main()

2026-07-04 08:59:07 [INFO] Starting OpenLibrary data download...
2026-07-04 08:59:07 [INFO] Searching for: 'harry potter'
2026-07-04 08:59:07 [INFO] Fetching 'harry potter' - page 1/10...
2026-07-04 08:59:10 [INFO] Saved to Search\Title=harry_potter\Page=01\data.json
2026-07-04 08:59:10 [INFO] Fetching 'harry potter' - page 2/10...
2026-07-04 08:59:12 [INFO] Saved to Search\Title=harry_potter\Page=02\data.json
2026-07-04 08:59:13 [INFO] Fetching 'harry potter' - page 3/10...
2026-07-04 08:59:15 [INFO] Saved to Search\Title=harry_potter\Page=03\data.json
2026-07-04 08:59:15 [INFO] Fetching 'harry potter' - page 4/10...
2026-07-04 08:59:18 [INFO] Saved to Search\Title=harry_potter\Page=04\data.json
2026-07-04 08:59:18 [INFO] Fetching 'harry potter' - page 5/10...
2026-07-04 08:59:20 [INFO] Saved to Search\Title=harry_potter\Page=05\data.json
2026-07-04 08:59:20 [INFO] Fetching 'harry potter' - page 6/10...
2026-07-04 08:59:22 [INFO] Saved to Search\Title=harry_potter\Page=06\data.json
20